<a href="https://colab.research.google.com/github/AKsToolBox/AKsToolBox/blob/main/colab_n8n_coder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# 4. Launch an authenticated OpenAI-compatible proxy + Cloudflare Quick Tunnel
import os
import re
import secrets
import subprocess
import threading
import time
import httpx
import uvicorn
from fastapi import FastAPI, Header, HTTPException, Request
from fastapi.responses import Response, StreamingResponse

API_KEY = os.environ.get("COLAB_API_KEY") or f"sk-colab-{secrets.token_urlsafe(18)}"
OLLAMA_BASE = "http://127.0.0.1:11434"

app = FastAPI()

@app.get("/health")
async def health():
    return {"status": "ok", "model": MODEL_ID}

@app.api_route("/v1/{path:path}", methods=["GET", "POST", "OPTIONS"])
async def proxy_openai(path: str, request: Request, authorization: str = Header(default="")):
    if request.method != "OPTIONS" and authorization != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Invalid API key")

    url = f"{OLLAMA_BASE}/v1/{path}"
    body = await request.body()
    headers = {"content-type": request.headers.get("content-type", "application/json")}

    is_stream = b'"stream":true' in body.replace(b" ", b"")

    if is_stream:
        client = httpx.AsyncClient(timeout=None)
        req = client.build_request(request.method, url, content=body, headers=headers)
        upstream = await client.send(req, stream=True)

        async def stream_gen():
            try:
                async for chunk in upstream.aiter_bytes():
                    yield chunk
            finally:
                await upstream.aclose()
                await client.aclose()

        return StreamingResponse(
            stream_gen(),
            status_code=upstream.status_code,
            media_type=upstream.headers.get("content-type", "text/event-stream"),
        )

    async with httpx.AsyncClient(timeout=300.0) as client:
        resp = await client.request(request.method, url, content=body, headers=headers)
        return Response(
            content=resp.content,
            status_code=resp.status_code,
            media_type=resp.headers.get("content-type", "application/json")
        )

threading.Thread(
    target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"),
    daemon=True,
).start()
time.sleep(2)

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

PUBLIC_URL = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        PUBLIC_URL = match.group(0)
        break
    time.sleep(0.5)

if not PUBLIC_URL:
    raise RuntimeError("Could not obtain Cloudflare tunnel URL.")

print("=== Paste these into n8n (OpenAI Chat Model Credential) ===")
print("Base URL  :", f"{PUBLIC_URL}/v1")
print("API Key   :", API_KEY)
print("Model Name:", MODEL_ID)

ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use


=== Paste these into n8n (OpenAI Chat Model Credential) ===
Base URL  : https://crest-chains-herald-glory.trycloudflare.com/v1
API Key   : sk-colab-Copd1fOlOUAD0rFUlxm9Fad8


NameError: name 'MODEL_ID' is not defined

In [ ]:
# 4. Launch an authenticated OpenAI-compatible proxy + Cloudflare Quick Tunnel
import os
import re
import secrets
import subprocess
import threading
import time
import httpx
import uvicorn
from fastapi import FastAPI, Header, HTTPException, Request
from fastapi.responses import Response, StreamingResponse

API_KEY = os.environ.get("COLAB_API_KEY") or f"sk-colab-{secrets.token_urlsafe(18)}"
OLLAMA_BASE = "http://127.0.0.1:11434"

app = FastAPI()

@app.get("/health")
async def health():
    return {"status": "ok", "model": MODEL_ID}

@app.api_route("/v1/{path:path}", methods=["GET", "POST", "OPTIONS"])
async def proxy_openai(path: str, request: Request, authorization: str = Header(default="")):
    if request.method != "OPTIONS" and authorization != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Invalid API key")

    url = f"{OLLAMA_BASE}/v1/{path}"
    body = await request.body()
    headers = {"content-type": request.headers.get("content-type", "application/json")}

    is_stream = b'"stream":true' in body.replace(b" ", b"")

    if is_stream:
        client = httpx.AsyncClient(timeout=None)
        req = client.build_request(request.method, url, content=body, headers=headers)
        upstream = await client.send(req, stream=True)

        async def stream_gen():
            try:
                async for chunk in upstream.aiter_bytes():
                    yield chunk
            finally:
                await upstream.aclose()
                await client.aclose()

        return StreamingResponse(
            stream_gen(),
            status_code=upstream.status_code,
            media_type=upstream.headers.get("content-type", "text/event-stream"),
        )

    async with httpx.AsyncClient(timeout=300.0) as client:
        resp = await client.request(request.method, url, content=body, headers=headers)
        return Response(
            content=resp.content,
            status_code=resp.status_code,
            media_type=resp.headers.get("content-type", "application/json")
        )

threading.Thread(
    target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"),
    daemon=True,
).start()
time.sleep(2)

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

PUBLIC_URL = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        PUBLIC_URL = match.group(0)
        break
    time.sleep(0.5)

if not PUBLIC_URL:
    raise RuntimeError("Could not obtain Cloudflare tunnel URL.")

print("=== Paste these into n8n (OpenAI Chat Model Credential) ===")
print("Base URL  :", f"{PUBLIC_URL}/v1")
print("API Key   :", API_KEY)
print("Model Name:", MODEL_ID)

# Colab T4 OpenAI-Compatible Coder & Agent Server (Ollama + Auth Proxy + Cloudflare)
Designed for testing **n8n OpenAI Chat Model** nodes and agentic tool-calling workflows on a free 15 GB Tesla T4.- Default model: `qwen3:14b` (~9.3 GB Q4_K_M, strong coding + reliable OpenAI `tool_calls`).- Uses `q8_0` KV cache + Flash Attention to fit long contexts comfortably on a T4.- Includes Bearer API key authentication and full SSE streaming support.

In [ ]:
# 1. Verify that a T4 GPU (15 GB VRAM) is attached!nvidia-smi

In [ ]:
# 2. Install Ollama, cloudflared, and lightweight proxy dependencies
# We skip broken repositories to prevent apt-get from hanging
!apt-get update -o Acquire::Retries=3 -qq || true
!apt-get install -y -qq zstd pciutils curl > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!pip install -q fastapi uvicorn httpx

# Verify ollama is installed and print its path
import shutil
ollama_path = shutil.which("ollama")
if ollama_path:
    print(f"Ollama successfully installed at: {ollama_path}")
else:
    print("Warning: Ollama installation could not be found in PATH.")

In [ ]:
# 3. Start Ollama with T4-friendly VRAM settings and pull the model
import os
import subprocess
import time
import urllib.request

# Best fit for T4 (15 GB VRAM) with coding + n8n tool calling:
MODEL_ID = "qwen3:14b"
# Swap to "qwen2.5-coder:14b" if you only want pure code generation without tool calling.

ollama_env = os.environ.copy()
ollama_env.update({
    "OLLAMA_HOST": "127.0.0.1:11434",
    "OLLAMA_FLASH_ATTENTION": "1",
    "OLLAMA_KV_CACHE_TYPE": "q8_0",
    "OLLAMA_KEEP_ALIVE": "-1",
    "OLLAMA_NUM_PARALLEL": "2",
    "OLLAMA_CONTEXT_LENGTH": "16384"
})

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    env=ollama_env,
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT
)

for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2) as r:
            if r.status == 200:
                break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama failed to start. Check /tmp/ollama.log")

print(f"Pulling {MODEL_ID} (this takes ~1-2 minutes on Colab)...")
subprocess.run(["ollama", "pull", MODEL_ID], check=True)
print(f"Model ready: {MODEL_ID}")

In [ ]:
# 4. Launch an authenticated OpenAI-compatible proxy + Cloudflare Quick Tunnel
import os
import re
import secrets
import subprocess
import threading
import time
import httpx
import uvicorn
from fastapi import FastAPI, Header, HTTPException, Request
from fastapi.responses import Response, StreamingResponse

API_KEY = os.environ.get("COLAB_API_KEY") or f"sk-colab-{secrets.token_urlsafe(18)}"
OLLAMA_BASE = "http://127.0.0.1:11434"

app = FastAPI()

@app.get("/health")
async def health():
    return {"status": "ok", "model": MODEL_ID}

@app.api_route("/v1/{path:path}", methods=["GET", "POST", "OPTIONS"])
async def proxy_openai(path: str, request: Request, authorization: str = Header(default="")):
    if request.method != "OPTIONS" and authorization != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Invalid API key")

    url = f"{OLLAMA_BASE}/v1/{path}"
    body = await request.body()
    headers = {"content-type": request.headers.get("content-type", "application/json")}

    is_stream = b'"stream":true' in body.replace(b" ", b"")

    if is_stream:
        client = httpx.AsyncClient(timeout=None)
        req = client.build_request(request.method, url, content=body, headers=headers)
        upstream = await client.send(req, stream=True)

        async def stream_gen():
            try:
                async for chunk in upstream.aiter_bytes():
                    yield chunk
            finally:
                await upstream.aclose()
                await client.aclose()

        return StreamingResponse(
            stream_gen(),
            status_code=upstream.status_code,
            media_type=upstream.headers.get("content-type", "text/event-stream"),
        )

    async with httpx.AsyncClient(timeout=300.0) as client:
        resp = await client.request(request.method, url, content=body, headers=headers)
        return Response(
            content=resp.content,
            status_code=resp.status_code,
            media_type=resp.headers.get("content-type", "application/json")
        )

# Check if server is already running to avoid [Errno 98]
server_up = False
try:
    if httpx.get("http://127.0.0.1:8000/health", timeout=1).status_code == 200:
        server_up = True
except:
    pass

if not server_up:
    threading.Thread(
        target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"),
        daemon=True,
    ).start()
    time.sleep(2)

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

PUBLIC_URL = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        PUBLIC_URL = match.group(0)
        break
    time.sleep(0.5)

if not PUBLIC_URL:
    raise RuntimeError("Could not obtain Cloudflare tunnel URL.")

print("=== Paste these into n8n (OpenAI Chat Model Credential) ===")
print("Base URL  :", f"{PUBLIC_URL}/v1")
print("API Key   :", API_KEY)
print("Model Name:", MODEL_ID)

In [ ]:
# 5. Self-test: verify public tunnel, Bearer auth, and OpenAI tool calling
import json
import httpx
import time

print("Pre-warming the model (loading weights into GPU)... This may take up to 2 minutes on first run.")
try:
    # Send a tiny prompt directly to Ollama first to warm up the weights
    httpx.post(
        f"{OLLAMA_BASE}/api/generate",
        json={"model": MODEL_ID, "prompt": "hello", "keep_alive": -1},
        timeout=180.0
    )
    print("Model loaded into GPU successfully!")
except Exception as e:
    print(f"Pre-warm notice (continuing anyway): {e}")

payload = {
    "model": MODEL_ID,
    "temperature": 0,
    "messages": [
        {"role": "user", "content": "Check the status of service billing-api using the tool."}
    ],
    "tools": [
        {
            "type": "function",
            "function": {
                "name": "check_service_status",
                "description": "Checks the health status of a given backend service",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "service_name": {"type": "string"}
                    },
                    "required": ["service_name"]
                }
            }
        }
    ]
}

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

print("Sending agent self-test query...")
resp = httpx.post(
    f"http://127.0.0.1:8000/v1/chat/completions",
    json=payload,
    headers=headers,
    timeout=120.0
)

print("Status code:", resp.status_code)
print("Response:", json.dumps(resp.json(), indent=2))